# Notebook 03 — Ablation study and data-leakage experiment

**Requires:** Notebook 01 (splits) and Notebook 02 (at least `resnet50__P`, `convnext_base__B` finished: they are the references).
**Runtime:** GPU (T4). **Expected time:** ≈ 4 h (ablation ≈ 3 h, leakage ≈ 1 h), based on the Notebook 02 timings
(ResNet-50 ≈ 3.8 min per fold, ConvNeXt-Base ≈ 8.8 min per fold). It can be split over several sessions.
Same safety as Notebook 02: finished folds are skipped, one overwritten checkpoint per model, Run all to resume.

### Part A — Ablation of the training recipe (Reviewer 4, comment 4)
ResNet-50 with recipe **P** is the reference (trained in Notebook 02, same folds, same seeds). Each run below changes
**one** component, so the difference to the reference is the contribution of that component.

| Configuration | What is changed relative to recipe P |
|---|---|
| `resnet50__P-no_aug` | no data augmentation (resize + normalise only) |
| `resnet50__P-no_imbalance` | no balanced sampler **and** no class weights |
| `resnet50__P-loss_weights_only` | class-weighted loss kept, balanced sampler removed |
| `resnet50__P-sampler_only` | balanced sampler kept, class weights removed |
| `resnet50__P-no_scheduler` | constant learning rate 1e-3 instead of OneCycleLR |
| `resnet50__P-cosine` | cosine annealing instead of OneCycleLR |
| `resnet50__P-full_finetune` | all layers trained (learning rate 1e-4, because 1e-3 on all layers destroys the pretrained features) |
| `resnet50__P-linear_head` | single linear layer instead of the 512-unit head |
| `resnet50__vanilla_TL` | plain transfer learning: no augmentation, no imbalance handling, constant LR, linear head |

All ablation runs are also tested under the simulated field degradations (does augmentation buy robustness? — Reviewer 1).

### Part B — How much does a leaky split inflate accuracy? (Reviewer 4, comment 6)
The two references are re-trained on a **naive** stratified 5-fold split that ignores the duplicate groups
(this mimics a split by hand or by random sampling, as in the first submission). Comparing naive vs group-aware
results, and accuracy on test images that **do** vs **do not** have a near-copy in training, measures the inflation.

| Configuration | Split |
|---|---|
| `resnet50__P__naive_split` | naive stratified 5-fold (duplicates may cross train/test) |
| `convnext_base__B__naive_split` | naive stratified 5-fold |

**When finished:** send me `NB03_results.zip` (it includes `research_log_NB03.md` with every number at full precision).

## 0. Settings

In [1]:
import os
NOTEBOOK, NB_TAG = "03_Ablation_and_Leakage", "NB03"
DRIVE_ROOT  = os.environ.get("PEANUT_DRIVE_ROOT", "/content/drive/MyDrive/Peanut_Revision")
LOCAL_ROOT  = os.environ.get("PEANUT_LOCAL_ROOT", "/content/peanut")

RUN_ABLATION = True
RUN_LEAKAGE  = True
REPEATS      = [0]
FOLDS        = None
CKPT_EVERY   = 5
NUM_WORKERS  = 2

TEST_MODE = os.environ.get("PEANUT_TEST") == "1"
if TEST_MODE:
    CKPT_EVERY, NUM_WORKERS, FOLDS = 2, 0, [0, 1]

## 1. Drive, GPU and environment

In [2]:
try:
    from google.colab import drive
    drive.mount("/content/drive")
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

import sys, json, time, shutil, zipfile, random, copy, platform, io, datetime as dt
from pathlib import Path
from dataclasses import dataclass, asdict, replace
import numpy as np, pandas as pd
import torch, torch.nn as nn, torchvision
import torchvision.transforms as T
import torchvision.models as M
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from PIL import Image, ImageEnhance, ImageFilter
import sklearn
from sklearn.metrics import (accuracy_score, balanced_accuracy_score, cohen_kappa_score, confusion_matrix,
                             f1_score, precision_recall_fscore_support, roc_auc_score)
from sklearn.utils.class_weight import compute_class_weight

D = {k: Path(DRIVE_ROOT) / v for k, v in {"data": "01_data", "splits": "02_splits", "results": "04_results",
     "checkpoints": "05_checkpoints", "models": "06_models", "logs": "09_logs"}.items()}
for p in D.values(): p.mkdir(parents=True, exist_ok=True)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
AMP = DEVICE.type == "cuda"
GPU = torch.cuda.get_device_name(0) if AMP else "CPU"
if not AMP and not TEST_MODE:
    raise SystemExit("No GPU found. Runtime -> Change runtime type -> T4 GPU, then run all again.")
torch.backends.cudnn.benchmark = True

ENV = {"notebook": NOTEBOOK, "run_at": dt.datetime.now().isoformat(timespec="seconds"),
       "gpu": GPU, "python": platform.python_version(), "torch": torch.__version__,
       "torchvision": torchvision.__version__, "cuda": torch.version.cuda, "cudnn": torch.backends.cudnn.version(),
       "numpy": np.__version__, "scikit-learn": sklearn.__version__, "pillow": Image.__version__}
(D["logs"] / f"env_{NB_TAG}_{ENV['run_at'].replace(':', '-')}.json").write_text(json.dumps(ENV, indent=2))
print(json.dumps(ENV, indent=2))

Mounted at /content/drive
{
  "notebook": "03_Ablation_and_Leakage",
  "run_at": "2026-10-31T14:18:26",
  "gpu": "Tesla T4",
  "python": "3.13.16",
  "torch": "2.11.0+cu130",
  "torchvision": "0.26.0+cu130",
  "cuda": "13.0",
  "cudnn": 92700,
  "numpy": "2.1.3",
  "scikit-learn": "1.6.1",
  "pillow": "11.3.0"
}


## 2. Index, folds and images

In [3]:
CLASSES = ["Alternaria leaf spot", "Healthy", "Leaf spot (early and late)", "Rosette", "Rust"]
SHORT = ["Alternaria", "Healthy", "Leaf spot", "Rosette", "Rust"]

idx = pd.read_csv(D["splits"] / "index_original.csv")
cv = pd.read_csv(D["splits"] / "cv_splits.csv")
cv_meta = json.loads((D["splits"] / "cv_splits_meta.json").read_text())
print(f"{len(idx)} images | grouping: pHash <= {cv_meta['phash_thr']} and dHash <= {cv_meta['dhash_thr']}, "
      f"or {cv_meta.get('emb_model', '-')} cosine >= {cv_meta.get('emb_thr', '-')}, or combined (cos, pHash, dHash) {cv_meta.get('combo', '-')} "
      f"| {cv_meta['n_groups']} duplicate groups | {cv_meta['k']} folds")

# Guard: results in 04_results must come from the current splits.
sig_file = D["results"] / "split_signature.json"
if sig_file.exists() and json.loads(sig_file.read_text()) != cv_meta:
    raise SystemExit("04_results/ contains results trained on DIFFERENT splits.\n"
                     "Run Notebook 01 again (it archives old results automatically), then run this notebook again.")
sig_file.write_text(json.dumps(cv_meta, indent=2))

local_cache = Path(LOCAL_ROOT) / "cache448"
if not any(local_cache.rglob("*.jpg")):
    t0 = time.time()
    local_zip = Path(LOCAL_ROOT) / "groundnut_v2_cache448.zip"
    local_zip.parent.mkdir(parents=True, exist_ok=True)
    shutil.copyfile(D["data"] / "groundnut_v2_cache448.zip", local_zip)
    with zipfile.ZipFile(local_zip) as z:
        z.extractall(LOCAL_ROOT)
    local_zip.unlink()
    print(f"cache copied and unzipped in {time.time()-t0:.0f}s")

t0 = time.time()
IMAGES = {iid: np.asarray(Image.open(local_cache / rel).convert("RGB")) for iid, rel in zip(idx["image_id"], idx["cache_rel"])}
LABELS = dict(zip(idx["image_id"], idx["label"]))
print(f"{len(IMAGES)} images loaded into RAM in {time.time()-t0:.0f}s "
      f"({sum(a.nbytes for a in IMAGES.values())/1e9:.2f} GB)")

1720 images | grouping: pHash <= 12 and dHash <= 16, or dinov2_vits14 cosine >= 0.9694550037384033, or combined (cos, pHash, dHash) [0.95, 20, 20] | 1112 duplicate groups | 5 folds
cache copied and unzipped in 3s
1720 images loaded into RAM in 4s (1.04 GB)


## 3. Recipes and models

In [4]:
@dataclass
class Config:
    name: str
    backbone: str
    recipe: str
    aug: str = "A"
    sampler: bool = True
    class_weighted_loss: bool = True
    label_smoothing: float = 0.0
    optimizer: str = "adam"
    lr: float = 1e-3
    weight_decay: float = 1e-5
    scheduler: str = "onecycle"
    epochs: int = 25
    finetune: str = "partial"
    head: str = "mlp"
    batch_size: int = 32
    img_size: int = 224
    select_metric: str = "val_macro_f1"
    pretrained: bool = True
    split: str = "cv_splits"          # which split file in 02_splits/ (cv_splits = group-aware)
    notes: str = ""

def recipe_P(backbone, **kw):
    big = backbone == "efficientnet_b4"
    kw.setdefault("name", f"{backbone}__P")
    return Config(backbone=backbone, recipe="P", img_size=380 if big else 224, batch_size=16 if big else 32, **kw)

def recipe_B(backbone, **kw):
    big = backbone == "efficientnet_b4"
    kw.setdefault("name", f"{backbone}__B")
    return Config(backbone=backbone, recipe="B", aug="B", sampler=False, class_weighted_loss=False,
                  label_smoothing=0.2, optimizer="adamw", lr=1e-5, weight_decay=1e-4, scheduler="cosine_wr",
                  finetune="full", head="linear", img_size=380 if big else 224, batch_size=16 if big else 32, **kw)

BACKBONES = ["resnet50", "efficientnet_b0", "efficientnet_b4", "convnext_base"]
MEAN, STD = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]

def get_transforms(aug, size):
    if aug == "A":    # Disease_CNN_Transfer_learning_RasNet_50.ipynb
        train = T.Compose([T.Resize((size, size)), T.RandomHorizontalFlip(), T.RandomVerticalFlip(), T.RandomRotation(20),
                           T.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.3),
                           T.RandomAffine(degrees=0, translate=(0.1, 0.1)), T.ToTensor(), T.RandomErasing(p=0.2),
                           T.Normalize(MEAN, STD)])
    elif aug == "B":  # Disease-EfficientNet-B0/B4, Disease-ConvNeXt-Base
        train = T.Compose([T.RandomHorizontalFlip(p=0.5), T.RandomRotation(15), T.RandomResizedCrop(size, scale=(0.7, 1.0)),
                           T.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.3),
                           T.RandomAffine(degrees=0, translate=(0.1, 0.1)), T.RandomGrayscale(p=0.1), T.ToTensor(),
                           T.Normalize(MEAN, STD)])
    elif aug == "none":
        train = T.Compose([T.Resize((size, size)), T.ToTensor(), T.Normalize(MEAN, STD)])
    else:
        raise ValueError(aug)
    test = T.Compose([T.Resize((size, size)), T.ToTensor(), T.Normalize(MEAN, STD)])
    return train, test

def build_model(cfg, num_classes=5):
    def head(in_f):
        if cfg.head == "mlp":
            return nn.Sequential(nn.Linear(in_f, 512), nn.ReLU(), nn.Dropout(0.3), nn.Linear(512, num_classes))
        return nn.Linear(in_f, num_classes)
    b, pre = cfg.backbone, cfg.pretrained
    if b == "resnet50":
        m = M.resnet50(weights=M.ResNet50_Weights.IMAGENET1K_V1 if pre else None)
        m.fc = head(m.fc.in_features); trainable = ["layer4.", "fc."]
    elif b == "efficientnet_b0":
        m = M.efficientnet_b0(weights=M.EfficientNet_B0_Weights.IMAGENET1K_V1 if pre else None)
        m.classifier[1] = head(m.classifier[1].in_features); trainable = ["features.6.", "features.7.", "features.8.", "classifier."]
    elif b == "efficientnet_b4":
        m = M.efficientnet_b4(weights=M.EfficientNet_B4_Weights.IMAGENET1K_V1 if pre else None)
        m.classifier[1] = head(m.classifier[1].in_features); trainable = ["features.6.", "features.7.", "features.8.", "classifier."]
    elif b == "convnext_base":
        m = M.convnext_base(weights=M.ConvNeXt_Base_Weights.IMAGENET1K_V1 if pre else None)
        m.classifier[2] = head(m.classifier[2].in_features); trainable = ["features.6.", "features.7.", "classifier."]
    else:
        raise ValueError(b)
    if cfg.finetune == "partial":
        for n, p in m.named_parameters():
            p.requires_grad = any(n.startswith(t) for t in trainable)
    return m

def show_configs(configs):
    for cfg in configs:
        m = build_model(replace(cfg, pretrained=False))
        tr = sum(p.numel() for p in m.parameters() if p.requires_grad); tot = sum(p.numel() for p in m.parameters())
        print(f"{cfg.name:<36} trainable {tr/1e6:6.2f} M of {tot/1e6:6.2f} M | input {cfg.img_size} | batch {cfg.batch_size} | split {cfg.split}")
        del m

In [5]:
# ---- simulated field degradations (applied to the 448-px image before resizing) ----
def _noise(img, std, seed):
    a = np.asarray(img).astype(np.float32) / 255.0
    a = a + np.random.default_rng(seed).normal(0, std, a.shape)
    return Image.fromarray((np.clip(a, 0, 1) * 255).astype(np.uint8))

def _occlude(img, frac, seed):
    rng = np.random.default_rng(seed); a = np.asarray(img).copy(); h, w = a.shape[:2]
    s = int(np.sqrt(frac) * min(h, w)); y, x = rng.integers(0, h - s), rng.integers(0, w - s)
    a[y:y + s, x:x + s] = 0
    return Image.fromarray(a)

def _shadow(img, factor, seed):
    rng = np.random.default_rng(seed); a = np.asarray(img).astype(np.float32); w = a.shape[1]
    x0 = rng.integers(0, w // 2); a[:, x0:x0 + w // 2] *= factor
    return Image.fromarray(np.clip(a, 0, 255).astype(np.uint8))

def _jpeg(img, q):
    buf = io.BytesIO(); img.save(buf, format="JPEG", quality=q); buf.seek(0)
    return Image.open(buf).convert("RGB")

PERTURBATIONS = {
    "clean":            lambda im, s: im,
    "dark_0.6":         lambda im, s: ImageEnhance.Brightness(im).enhance(0.6),
    "overexposed_1.4":  lambda im, s: ImageEnhance.Brightness(im).enhance(1.4),
    "low_contrast_0.6": lambda im, s: ImageEnhance.Contrast(im).enhance(0.6),
    "hard_shadow_0.4":  lambda im, s: _shadow(im, 0.4, s),
    "blur_sigma2":      lambda im, s: im.filter(ImageFilter.GaussianBlur(2)),
    "noise_sd0.05":     lambda im, s: _noise(im, 0.05, s),
    "jpeg_q20":         lambda im, s: _jpeg(im, 20),
    "occlusion_25pct":  lambda im, s: _occlude(im, 0.25, s),
    "rotation_45":      lambda im, s: im.rotate(45),
}

## 4. Training code (identical to Notebook 02)

In [6]:
SPLITS = {"cv_splits": cv}            # notebooks can add more split tables (e.g. the naive split in Notebook 03)

class ImageSet(Dataset):
    def __init__(self, ids, tf, perturb=None):
        self.ids, self.tf, self.perturb = list(ids), tf, perturb
    def __len__(self): return len(self.ids)
    def __getitem__(self, i):
        img = Image.fromarray(IMAGES[self.ids[i]])
        if self.perturb is not None:
            img = self.perturb(img, i)
        return self.tf(img), LABELS[self.ids[i]]

def seed_everything(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)

def metrics(y, p, prob=None):
    lab = list(range(len(CLASSES)))
    pr, rc, f1, sup = precision_recall_fscore_support(y, p, labels=lab, zero_division=0)
    out = {"accuracy": accuracy_score(y, p), "n_correct": int((np.asarray(y) == np.asarray(p)).sum()), "n": int(len(y)),
           "balanced_accuracy": balanced_accuracy_score(y, p),
           "macro_precision": float(pr.mean()), "macro_recall": float(rc.mean()),
           "macro_f1": f1_score(y, p, average="macro", labels=lab, zero_division=0),
           "weighted_f1": f1_score(y, p, average="weighted", labels=lab, zero_division=0),
           "cohen_kappa": cohen_kappa_score(y, p),
           "per_class": {CLASSES[i]: {"precision": float(pr[i]), "recall": float(rc[i]), "f1": float(f1[i]),
                                      "support": int(sup[i])} for i in lab},
           "confusion_matrix": confusion_matrix(y, p, labels=lab).tolist()}
    if prob is not None and len(np.unique(y)) == len(lab):
        out["macro_auc_ovr"] = roc_auc_score(y, prob, multi_class="ovr", labels=lab)
    return {k: (float(v) if isinstance(v, (float, np.floating)) else v) for k, v in out.items()}

@torch.no_grad()
def predict(model, loader):
    model.eval(); ys, ps, probs, loss, n = [], [], [], 0.0, 0
    ce = nn.CrossEntropyLoss(reduction="sum")
    for x, y in loader:
        x, y = x.to(DEVICE, non_blocking=True), y.to(DEVICE)
        with torch.autocast(DEVICE.type, dtype=torch.float16, enabled=AMP):
            out = model(x)
        out = out.float(); loss += ce(out, y).item(); n += len(y)
        probs.append(torch.softmax(out, 1).cpu().numpy()); ps.append(out.argmax(1).cpu().numpy()); ys.append(y.cpu().numpy())
    return np.concatenate(ys), np.concatenate(ps), np.concatenate(probs), loss / max(n, 1)

def ckpt_path(cfg): return D["checkpoints"] / f"{cfg.name}.pt"

def save_checkpoint(cfg, r, f, epoch, model, opt, sched, scaler, sampler_gen, best, best_state, best_epoch, log):
    state = {"config": cfg.name, "repeat": r, "fold": f, "epoch": epoch, "model": model.state_dict(),
             "opt": opt.state_dict(), "sched": sched.state_dict() if sched else None, "scaler": scaler.state_dict(),
             "sampler_gen": sampler_gen.get_state(), "best": best, "best_epoch": best_epoch,
             "best_state": {k: (v.half() if v.is_floating_point() else v) for k, v in best_state.items()},
             "log": log, "rng": {"py": random.getstate(), "np": np.random.get_state(), "torch": torch.get_rng_state(),
                                 "cuda": torch.cuda.get_rng_state_all() if AMP else None}}
    torch.save(state, ckpt_path(cfg))      # same file every time: overwritten in place, never sent to the Drive bin
    if os.environ.get("PEANUT_CRASH_AFTER_EPOCH") == str(epoch):   # smoke-test hook only
        raise KeyboardInterrupt("simulated disconnect")

def load_checkpoint(cfg, r, f):
    p = ckpt_path(cfg)
    if not p.exists() or p.stat().st_size == 0: return None
    try:
        st = torch.load(p, map_location="cpu", weights_only=False)
    except Exception as e:
        print("  checkpoint unreadable, starting this fold from scratch:", e); return None
    if (st["config"], st["repeat"], st["fold"]) != (cfg.name, r, f): return None
    return st

def clear_checkpoint(cfg):
    p = ckpt_path(cfg)
    if p.exists() and p.stat().st_size > 0:
        open(p, "wb").close()             # empty it (keeps one 0-byte file instead of filling the bin)

def run_fold(cfg, r, f, robustness=False, save_weights=False):
    r, f = int(r), int(f)                 # plain Python ints (numpy ints break random.seed and JSON)
    run_dir = D["results"] / cfg.name / f"r{r}_f{f}"
    if (run_dir / "metrics.json").exists():
        print(f"[{cfg.name}] repeat {r} fold {f}: already done"); return json.loads((run_dir / "metrics.json").read_text())
    run_dir.mkdir(parents=True, exist_ok=True)
    sp = SPLITS[cfg.split]
    s = sp[(sp["repeat"] == r) & (sp["fold"] == f)]
    ids = {role: s.loc[s["role"] == role, "image_id"].tolist() for role in ("train", "val", "test")}
    seed = 1000 * r + f                      # same seed for every configuration -> paired comparisons
    seed_everything(seed)
    tf_train, tf_test = get_transforms(cfg.aug, cfg.img_size)
    y_tr = np.array([LABELS[i] for i in ids["train"]])
    gen = torch.Generator().manual_seed(seed)
    if cfg.sampler:
        w = (1.0 / np.bincount(y_tr, minlength=len(CLASSES)))[y_tr]
        dl_tr = DataLoader(ImageSet(ids["train"], tf_train), batch_size=cfg.batch_size, num_workers=NUM_WORKERS,
                           sampler=WeightedRandomSampler(torch.tensor(w, dtype=torch.double), len(w), True, generator=gen),
                           pin_memory=AMP, persistent_workers=NUM_WORKERS > 0)
    else:
        dl_tr = DataLoader(ImageSet(ids["train"], tf_train), batch_size=cfg.batch_size, shuffle=True, generator=gen,
                           num_workers=NUM_WORKERS, pin_memory=AMP, persistent_workers=NUM_WORKERS > 0)
    dl_va = DataLoader(ImageSet(ids["val"], tf_test), batch_size=64, num_workers=NUM_WORKERS, pin_memory=AMP)
    dl_te = DataLoader(ImageSet(ids["test"], tf_test), batch_size=64, num_workers=NUM_WORKERS, pin_memory=AMP)

    model = build_model(cfg).to(DEVICE)
    cw = None
    if cfg.class_weighted_loss:
        cw = torch.tensor(compute_class_weight("balanced", classes=np.arange(len(CLASSES)), y=y_tr), dtype=torch.float, device=DEVICE)
    criterion = nn.CrossEntropyLoss(weight=cw, label_smoothing=cfg.label_smoothing)
    params = [p for p in model.parameters() if p.requires_grad]
    opt = (torch.optim.Adam if cfg.optimizer == "adam" else torch.optim.AdamW)(params, lr=cfg.lr, weight_decay=cfg.weight_decay)
    per_batch = cfg.scheduler == "onecycle"
    sched = {"onecycle": lambda: torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=cfg.lr, steps_per_epoch=len(dl_tr), epochs=cfg.epochs),
             "cosine_wr": lambda: torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(opt, T_0=5, T_mult=2),
             "cosine": lambda: torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=cfg.epochs),
             "none": lambda: None}[cfg.scheduler]()
    scaler = torch.amp.GradScaler("cuda", enabled=AMP)

    start, best, best_state, best_epoch, log = 0, (-1.0, -1.0), None, 0, []
    st = load_checkpoint(cfg, r, f)
    if st:
        model.load_state_dict(st["model"]); opt.load_state_dict(st["opt"]); scaler.load_state_dict(st["scaler"])
        if sched and st["sched"]: sched.load_state_dict(st["sched"])
        gen.set_state(st["sampler_gen"]); best, best_epoch, log = tuple(st["best"]), st["best_epoch"], st["log"]
        best_state = {k: v.float() if v.is_floating_point() else v for k, v in st["best_state"].items()}
        random.setstate(st["rng"]["py"]); np.random.set_state(st["rng"]["np"]); torch.set_rng_state(st["rng"]["torch"])
        if AMP and st["rng"]["cuda"]: torch.cuda.set_rng_state_all(st["rng"]["cuda"])
        start = st["epoch"]
        print(f"  resumed from checkpoint after epoch {start}")
        del st

    print(f"[{cfg.name}] repeat {r} fold {f}: train {len(ids['train'])}, val {len(ids['val'])}, test {len(ids['test'])}")
    t_fold = time.time()
    for ep in range(start, cfg.epochs):
        t0 = time.time(); model.train(); tl, tc, tn = 0.0, 0, 0
        for x, y in dl_tr:
            x, y = x.to(DEVICE, non_blocking=True), y.to(DEVICE, non_blocking=True)
            opt.zero_grad(set_to_none=True)
            with torch.autocast(DEVICE.type, dtype=torch.float16, enabled=AMP):
                out = model(x); loss = criterion(out, y)
            scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
            if per_batch: sched.step()
            tl += loss.item() * len(y); tc += (out.argmax(1) == y).sum().item(); tn += len(y)
        if sched is not None and not per_batch: sched.step()
        yv, pv, _, vloss = predict(model, dl_va); vm = metrics(yv, pv)
        row = {"epoch": ep + 1, "train_loss": tl / tn, "train_acc": tc / tn, "val_loss": vloss, "val_acc": vm["accuracy"],
               "val_macro_f1": vm["macro_f1"], "lr": opt.param_groups[0]["lr"], "time_s": time.time() - t0}
        log.append(row)
        score = (row[cfg.select_metric], -vloss)
        mark = ""
        if score > best:
            best, best_epoch, mark = score, ep + 1, " *"
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        print(f"  ep {ep+1:02d}  loss {row['train_loss']:.4f}  train acc {row['train_acc']:.4f}  "
              f"val acc {row['val_acc']:.4f}  val F1 {row['val_macro_f1']:.4f}  ({row['time_s']:.0f}s){mark}")
        if CKPT_EVERY and (ep + 1) % CKPT_EVERY == 0 and ep + 1 < cfg.epochs:
            save_checkpoint(cfg, r, f, ep + 1, model, opt, sched, scaler, gen, best, best_state, best_epoch, log)

    model.load_state_dict(best_state)
    yt, pt, prob, _ = predict(model, dl_te)
    m = metrics(yt, pt, prob)
    m.update({"config": asdict(cfg), "repeat": r, "fold": f, "seed": seed, "best_epoch": best_epoch,
              "best_val_macro_f1": best[0], "best_val_loss": -best[1],
              "train_minutes": (time.time() - t_fold) / 60, "n_train": len(ids["train"]), "n_val": len(ids["val"]),
              "n_test": len(ids["test"]), "gpu": GPU, "split_meta": cv_meta if cfg.split == "cv_splits" else cfg.split,
              "n_trainable_params": sum(p.numel() for p in model.parameters() if p.requires_grad),
              "n_total_params": sum(p.numel() for p in model.parameters())})
    pred = pd.DataFrame({"image_id": ids["test"], "y_true": yt, "y_pred": pt})
    for i in range(len(CLASSES)): pred[f"p_{i}"] = prob[:, i]

    if robustness:
        rob = {}
        for name, fn in PERTURBATIONS.items():
            dl = DataLoader(ImageSet(ids["test"], tf_test, perturb=fn), batch_size=64, num_workers=NUM_WORKERS)
            y_, p_, _, _ = predict(model, dl); mm = metrics(y_, p_)
            rob[name] = {"accuracy": mm["accuracy"], "n_correct": mm["n_correct"], "n": mm["n"], "macro_f1": mm["macro_f1"],
                         "per_class_recall": {c: mm["per_class"][c]["recall"] for c in CLASSES}}
        (run_dir / "robustness.json").write_text(json.dumps(rob, indent=2))
        m["robustness"] = {k: v["accuracy"] for k, v in rob.items()}
    if save_weights:
        wdir = D["models"] / cfg.name; wdir.mkdir(parents=True, exist_ok=True)
        torch.save({k: (v.half() if v.is_floating_point() else v) for k, v in best_state.items()}, wdir / f"r{r}_f{f}.pt")

    pd.DataFrame(log).to_csv(run_dir / "log.csv", index=False)
    pred.to_csv(run_dir / "predictions.csv", index=False)
    (run_dir / "metrics.json").write_text(json.dumps(m, indent=2))   # written last = the fold counts as done
    clear_checkpoint(cfg)
    print(f"[{cfg.name}] repeat {r} fold {f}: TEST accuracy {m['accuracy']:.4f} ({m['n_correct']}/{m['n']}), "
          f"macro-F1 {m['macro_f1']:.4f} (best epoch {best_epoch}, {m['train_minutes']:.1f} min)")
    del model, opt; torch.cuda.empty_cache() if AMP else None
    return m

def run_configs(configs, robustness_for=lambda c: False, weights_for=lambda c: False):
    for cfg in configs:
        (D["results"] / cfg.name).mkdir(parents=True, exist_ok=True)
        (D["results"] / cfg.name / "config.json").write_text(json.dumps(asdict(cfg), indent=2))
        sp = SPLITS[cfg.split]; folds_all = sorted(int(x) for x in sp["fold"].unique())
        t0 = time.time()
        for r in REPEATS:
            for f in (FOLDS if FOLDS is not None else folds_all):
                run_fold(cfg, r, f, robustness=robustness_for(cfg), save_weights=weights_for(cfg))
        done = all((D["results"] / cfg.name / f"r{r}_f{f}" / "metrics.json").exists() for r in REPEATS for f in folds_all)
        if done and ckpt_path(cfg).exists():
            ckpt_path(cfg).unlink()           # model finished: its (empty) checkpoint file is no longer needed
        print(f"==> {cfg.name} finished in {(time.time()-t0)/60:.1f} min\n")

## 5. The naive split (for Part B)
Same settings as the group-aware split (5 folds, 1/8 of the rest as validation, same seed) but **without** groups.
The cell also counts, per fold, how many test images have a near-duplicate in that fold's training/validation data.

In [7]:
from sklearn.model_selection import StratifiedKFold
NAIVE_CSV = D["splits"] / "cv_splits_naive.csv"
if NAIVE_CSV.exists():
    naive = pd.read_csv(NAIVE_CSV)
else:
    y = idx["label"].values; rows = []
    for r in range(cv_meta["repeats"]):
        outer = StratifiedKFold(n_splits=cv_meta["k"], shuffle=True, random_state=cv_meta["seed"] + r)
        for f, (trval, te) in enumerate(outer.split(np.zeros(len(y)), y)):
            inner = StratifiedKFold(n_splits=cv_meta["inner_val_folds"], shuffle=True, random_state=cv_meta["seed"] + 100 * r + f)
            tr_i, va_i = next(inner.split(np.zeros(len(trval)), y[trval]))
            role = np.empty(len(y), dtype=object)
            role[trval[tr_i]], role[trval[va_i]], role[te] = "train", "val", "test"
            rows.append(pd.DataFrame({"repeat": r, "fold": f, "image_id": idx["image_id"], "role": role}))
    naive = pd.concat(rows, ignore_index=True); naive.to_csv(NAIVE_CSV, index=False)
SPLITS["naive"] = naive

GROUP = dict(zip(idx["image_id"], idx["dup_group"]))
GSIZE = idx["dup_group"].map(idx["dup_group"].value_counts())
def leak_flags(split):
    # True for a test image whose duplicate group has another member in the same fold's train/val data
    out = {}
    for (r, f), s in split.groupby(["repeat", "fold"]):
        tv_groups = set(s.loc[s["role"] != "test", "image_id"].map(GROUP))
        for iid in s.loc[s["role"] == "test", "image_id"]:
            out[(int(r), int(f), iid)] = GROUP[iid] in tv_groups
    return out
LEAK_NAIVE, LEAK_GROUP = leak_flags(naive), leak_flags(cv)
lk = pd.Series(LEAK_NAIVE).groupby(level=[0, 1]).agg(["sum", "count"])
lk.columns = ["test images with a near-duplicate in train/val", "test images"]
print("NAIVE split:"); display(lk)
print("Group-aware split: test images with a near-duplicate in train/val =", int(sum(LEAK_GROUP.values())), "(must be 0)")

NAIVE split:


test images with a near-duplicate in train/val  test images
0 0                                             172          344
  1                                             182          344
  2                                             183          344
  3                                             189          344
  4                                             184          344

Group-aware split: test images with a near-duplicate in train/val = 0 (must be 0)


## 6. Configurations

In [8]:
P = recipe_P("resnet50")
ABLATION = [
    replace(P, name="resnet50__P-no_aug", aug="none"),
    replace(P, name="resnet50__P-no_imbalance", sampler=False, class_weighted_loss=False),
    replace(P, name="resnet50__P-loss_weights_only", sampler=False),
    replace(P, name="resnet50__P-sampler_only", class_weighted_loss=False),
    replace(P, name="resnet50__P-no_scheduler", scheduler="none"),
    replace(P, name="resnet50__P-cosine", scheduler="cosine"),
    replace(P, name="resnet50__P-full_finetune", finetune="full", lr=1e-4, notes="lr 1e-4 for full fine-tuning"),
    replace(P, name="resnet50__P-linear_head", head="linear"),
    replace(P, name="resnet50__vanilla_TL", aug="none", sampler=False, class_weighted_loss=False, scheduler="none",
            head="linear", notes="plain transfer learning"),
]
LEAKAGE = [replace(P, name="resnet50__P__naive_split", split="naive"),
           replace(recipe_B("convnext_base"), name="convnext_base__B__naive_split", split="naive")]
REFERENCES = ["resnet50__P", "convnext_base__B"]
if TEST_MODE:
    ABLATION = [replace(c, epochs=3, img_size=64, batch_size=8, pretrained=False) for c in ABLATION[:2]]
    LEAKAGE = [replace(c, epochs=3, img_size=64, batch_size=8, pretrained=False) for c in LEAKAGE]
for ref in REFERENCES:
    n = len(list((D["results"] / ref).glob("r*_f*/metrics.json")))
    print(f"reference {ref}: {n} folds found in 04_results" + ("" if n else "  <-- run Notebook 02 first"))
show_configs(ABLATION + LEAKAGE)

reference resnet50__P: 5 folds found in 04_results
reference convnext_base__B: 5 folds found in 04_results
resnet50__P-no_aug                   trainable  16.02 M of  24.56 M | input 224 | batch 32 | split cv_splits
resnet50__P-no_imbalance             trainable  16.02 M of  24.56 M | input 224 | batch 32 | split cv_splits
resnet50__P-loss_weights_only        trainable  16.02 M of  24.56 M | input 224 | batch 32 | split cv_splits
resnet50__P-sampler_only             trainable  16.02 M of  24.56 M | input 224 | batch 32 | split cv_splits
resnet50__P-no_scheduler             trainable  16.02 M of  24.56 M | input 224 | batch 32 | split cv_splits
resnet50__P-cosine                   trainable  16.02 M of  24.56 M | input 224 | batch 32 | split cv_splits
resnet50__P-full_finetune            trainable  24.56 M of  24.56 M | input 224 | batch 32 | split cv_splits
resnet50__P-linear_head              trainable  14.97 M of  23.52 M | input 224 | batch 32 | split cv_splits
resnet50__vanilla_TL 

## 7. Run Part A (ablation) and Part B (leakage)

In [9]:
if RUN_ABLATION:
    run_configs(ABLATION, robustness_for=lambda c: True)
if RUN_LEAKAGE:
    run_configs(LEAKAGE)

Downloading: "https://download.pytorch.org/models/resnet50-0676ba61.pth" to /root/.cache/torch/hub/checkpoints/resnet50-0676ba61.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 135MB/s]


[resnet50__P-no_aug] repeat 0 fold 0: train 1200, val 154, test 366
  ep 01  loss 0.8410  train acc 0.4158  val acc 0.7273  val F1 0.7416  (10s) *
  ep 02  loss 0.1837  train acc 0.9142  val acc 0.9221  val F1 0.9449  (5s) *
  ep 03  loss 0.0502  train acc 0.9717  val acc 0.9545  val F1 0.9668  (5s) *
  ep 04  loss 0.1391  train acc 0.9567  val acc 0.9286  val F1 0.9170  (5s)
  ep 05  loss 0.0461  train acc 0.9750  val acc 0.9221  val F1 0.9189  (5s)
  ep 06  loss 0.0718  train acc 0.9700  val acc 0.9416  val F1 0.9601  (5s)
  ep 07  loss 0.0449  train acc 0.9775  val acc 0.9545  val F1 0.9699  (5s) *
  ep 08  loss 0.0314  train acc 0.9817  val acc 0.9545  val F1 0.9684  (5s)
  ep 09  loss 0.1071  train acc 0.9683  val acc 0.9091  val F1 0.9221  (5s)
  ep 10  loss 0.0337  train acc 0.9817  val acc 0.9610  val F1 0.9718  (5s) *
  ep 11  loss 0.0232  train acc 0.9875  val acc 0.9675  val F1 0.9768  (5s) *
  ep 12  loss 0.0655  train acc 0.9875  val acc 0.9740  val F1 0.9820  (5s) *
  ep 

100%|██████████| 338M/338M [00:01<00:00, 180MB/s]


[convnext_base__B__naive_split] repeat 0 fold 0: train 1204, val 172, test 344
  ep 01  loss 1.4208  train acc 0.4651  val acc 0.7326  val F1 0.4521  (42s) *
  ep 02  loss 1.1282  train acc 0.7683  val acc 0.9186  val F1 0.7769  (16s) *
  ep 03  loss 0.9333  train acc 0.8978  val acc 0.9593  val F1 0.9170  (16s) *
  ep 04  loss 0.8574  train acc 0.9311  val acc 0.9709  val F1 0.9602  (16s) *
  ep 05  loss 0.8325  train acc 0.9468  val acc 0.9709  val F1 0.9602  (16s) *
  ep 06  loss 0.7967  train acc 0.9601  val acc 1.0000  val F1 1.0000  (16s) *
  ep 07  loss 0.7476  train acc 0.9743  val acc 0.9884  val F1 0.9911  (17s)
  ep 08  loss 0.7288  train acc 0.9842  val acc 0.9942  val F1 0.9956  (16s)
  ep 09  loss 0.7265  train acc 0.9817  val acc 0.9942  val F1 0.9956  (16s)
  ep 10  loss 0.7104  train acc 0.9875  val acc 0.9942  val F1 0.9956  (16s)
  ep 11  loss 0.7066  train acc 0.9892  val acc 0.9942  val F1 0.9956  (17s)
  ep 12  loss 0.7024  train acc 0.9884  val acc 0.9942  val F1

## 8. Results
* **Ablation:** each configuration minus the reference, fold by fold (same folds and seeds → paired differences).
* **Leakage:** group-aware vs naive split, and on the naive split the accuracy on test images *with* vs *without* a near-duplicate in training.

Tables are rounded for reading; `research_log_NB03.md` holds the exact values.

In [10]:
def load_runs(name):
    return [json.loads(p.read_text()) for p in sorted((D["results"] / name).glob("r*_f*/metrics.json"))]

def summary_table(names):
    rows = []
    for name in names:
        ms = load_runs(name)
        if not ms: continue
        row = {"configuration": name, "folds": len(ms)}
        for k in ["accuracy", "macro_precision", "macro_recall", "macro_f1", "balanced_accuracy", "cohen_kappa"]:
            v = 100 * np.array([m[k] for m in ms])
            row[k] = f"{v.mean():.2f} ± {v.std(ddof=1) if len(v) > 1 else 0:.2f}"
        for c, sname in zip(CLASSES, SHORT):
            row[f"recall {sname}"] = f"{100*np.mean([m['per_class'][c]['recall'] for m in ms]):.1f}"
        row["pooled correct"] = f"{sum(m['n_correct'] if 'n_correct' in m else int(np.trace(m['confusion_matrix'])) for m in ms)}/{sum(m['n_test'] for m in ms)}"
        row["min/fold"] = f"{np.mean([m['train_minutes'] for m in ms]):.1f}"
        rows.append(row)
    return pd.DataFrame(rows)

def robustness_table(names):
    rows = []
    for name in names:
        ms = [m for m in load_runs(name) if "robustness" in m]
        if ms:
            rows.append({"configuration": name, **{k: f"{100*np.mean([m['robustness'][k] for m in ms]):.1f}" for k in ms[0]["robustness"]}})
    return pd.DataFrame(rows).set_index("configuration").T if rows else None

def write_research_log(names, notebook, extra_sections=()):
    # Every number is written with repr() = the exact double stored in metrics.json (no rounding).
    R = repr
    L = [f"# Research log — {notebook}", "", f"Run: {ENV['run_at']} | GPU: {ENV['gpu']} | torch {ENV['torch']} | torchvision {ENV['torchvision']}",
         f"Split rule: pHash <= {cv_meta['phash_thr']} and dHash <= {cv_meta['dhash_thr']}, or same class and {cv_meta.get('emb_model', '-')} cosine >= {cv_meta.get('emb_thr', '-')}, or same class and (cosine, pHash, dHash) within {cv_meta.get('combo', '-')}; {cv_meta['n_groups']} groups; {cv_meta['k']} folds; seed {cv_meta['seed']}",
         "", "All values below are written programmatically from metrics.json / robustness.json with full precision (Python repr).",
         "Mean and SD are computed over folds (SD with ddof = 1) from the per-fold values listed.", ""]
    keys = ["accuracy", "balanced_accuracy", "macro_precision", "macro_recall", "macro_f1", "weighted_f1", "cohen_kappa", "macro_auc_ovr"]
    for name in names:
        ms = load_runs(name)
        if not ms: continue
        L += [f"## {name}", "", "Config: `" + json.dumps(ms[0]["config"]) + "`", "",
              "| fold | n_train | n_val | n_test | correct | " + " | ".join(keys) + " | best_epoch | best_val_macro_f1 | train_minutes |",
              "|" + "---|" * (8 + len(keys)) ]
        for m in ms:
            corr = m.get("n_correct", int(np.trace(m["confusion_matrix"])))
            L.append(f"| {m['fold']} | {m['n_train']} | {m['n_val']} | {m['n_test']} | {corr} | " +
                     " | ".join(R(m[k]) if k in m else "-" for k in keys) +
                     f" | {m['best_epoch']} | {R(m.get('best_val_macro_f1', '-'))} | {R(m['train_minutes'])} |")
        L += ["", "**Across folds**", ""]
        for k in keys:
            v = [m[k] for m in ms if k in m]
            if v:
                L.append(f"- {k}: mean {R(float(np.mean(v)))}, SD {R(float(np.std(v, ddof=1))) if len(v) > 1 else '-'}, min {R(min(v))}, max {R(max(v))}")
        tot_c = sum(m.get("n_correct", int(np.trace(m["confusion_matrix"]))) for m in ms); tot_n = sum(m["n_test"] for m in ms)
        L.append(f"- pooled out-of-fold: {tot_c}/{tot_n} correct = {R(tot_c / tot_n)}")
        pooled_cm = np.sum([np.array(m["confusion_matrix"]) for m in ms], axis=0)
        L += ["", "Pooled confusion matrix (rows = true, columns = predicted; order " + ", ".join(SHORT) + "):", "```", str(pooled_cm.tolist()), "```", ""]
        L += ["Per-class (per fold: precision / recall / F1 / support):", ""]
        L.append("| fold | " + " | ".join(SHORT) + " |"); L.append("|" + "---|" * (1 + len(CLASSES)))
        for m in ms:
            L.append(f"| {m['fold']} | " + " | ".join(f"{R(m['per_class'][c]['precision'])} / {R(m['per_class'][c]['recall'])} / {R(m['per_class'][c]['f1'])} / {m['per_class'][c]['support']}" for c in CLASSES) + " |")
        L += ["", "Per-class recall across folds: " + "; ".join(f"{s}: mean {R(float(np.mean([m['per_class'][c]['recall'] for m in ms])))}" for c, s in zip(CLASSES, SHORT)), ""]
        if any("robustness" in m for m in ms):
            perts = list(ms[0]["robustness"])
            L += ["Robustness (test accuracy under each degradation, per fold):", "", "| degradation | " + " | ".join(f"fold {m['fold']}" for m in ms) + " | mean |",
                  "|" + "---|" * (2 + len(ms))]
            for pz in perts:
                v = [m["robustness"][pz] for m in ms]
                L.append(f"| {pz} | " + " | ".join(R(x) for x in v) + f" | {R(float(np.mean(v)))} |")
            L.append("")
    for title, body in extra_sections:
        L += [f"## {title}", "", body, ""]
    path = D["logs"] / f"research_log_{NB_TAG}.md"
    path.write_text("\n".join(L))
    print("research log written:", path)
    return path

def make_results_zip(names, notebook, summary_df):
    out = Path(LOCAL_ROOT) / f"{NB_TAG}_results.zip"
    with zipfile.ZipFile(out, "w", zipfile.ZIP_DEFLATED) as z:
        for name in names:
            for p in (D["results"] / name).rglob("*"):
                if p.is_file() and p.suffix in {".json", ".csv"}:
                    z.write(p, f"04_results/{p.relative_to(D['results'])}")
        for p in D["logs"].glob(f"*{NB_TAG}*"):
            z.write(p, f"09_logs/{p.name}")
        for p in D["splits"].glob("*.json"):
            z.write(p, f"02_splits/{p.name}")
        if summary_df is not None:
            z.writestr("summary.csv", summary_df.to_csv(index=False))
    shutil.copyfile(out, D["results"] / out.name)
    print(f"Saved {D['results'] / out.name} ({out.stat().st_size/1e6:.1f} MB)")
    if IN_COLAB:
        from google.colab import files
        files.download(str(out))

ABL_NAMES = ["resnet50__P"] + [c.name for c in ABLATION]
LEAK_NAMES = [n for pair in zip(REFERENCES, [c.name for c in LEAKAGE]) for n in pair]
pd.set_option("display.width", 250); pd.set_option("display.max_columns", 30)
SUMMARY = summary_table(ABL_NAMES + [c.name for c in LEAKAGE] + ["convnext_base__B"])
display(SUMMARY)

def paired_delta(ref, names, key):
    refm = {m["fold"]: m[key] for m in load_runs(ref)}
    rows = []
    for n in names:
        mm = {m["fold"]: m[key] for m in load_runs(n)}
        common = sorted(set(refm) & set(mm))
        if not common: continue
        d = np.array([mm[f] - refm[f] for f in common])
        rows.append({"configuration": n, "folds": len(common), f"{key} mean diff (pts)": f"{100*d.mean():+.2f}",
                     "SD (pts)": f"{100*d.std(ddof=1) if len(d) > 1 else 0:.2f}",
                     "folds better / worse / equal": f"{int((d > 0).sum())} / {int((d < 0).sum())} / {int((d == 0).sum())}"})
    return pd.DataFrame(rows)

ABL_DELTA = pd.concat([paired_delta("resnet50__P", [c.name for c in ABLATION], "accuracy"),
                       paired_delta("resnet50__P", [c.name for c in ABLATION], "macro_f1")
                       .drop(columns=["folds", "configuration"])], axis=1)
print("Ablation: change relative to resnet50__P (negative = removing the component hurts)"); display(ABL_DELTA)
ROB = robustness_table(ABL_NAMES)
if ROB is not None:
    print("Accuracy (%) under simulated degradations:"); display(ROB)

leak_rows, leak_text = [], []
for ref, naive_cfg in zip(REFERENCES, [c.name for c in LEAKAGE]):
    for name, flags in [(ref, LEAK_GROUP), (naive_cfg, LEAK_NAIVE)]:
        preds = [pd.read_csv(p).assign(fold=int(p.parent.name.split("_f")[1]), repeat=int(p.parent.name[1:].split("_")[0]))
                 for p in sorted((D["results"] / name).glob("r*_f*/predictions.csv"))]
        if not preds: continue
        pr = pd.concat(preds)
        pr["leaked"] = [flags[(r, f, i)] for r, f, i in zip(pr["repeat"], pr["fold"], pr["image_id"])]
        pr["ok"] = pr["y_true"] == pr["y_pred"]
        row = {"configuration": name, "split": "naive" if name == naive_cfg else "group-aware",
               "all test images": f"{int(pr['ok'].sum())}/{len(pr)} = {100*pr['ok'].mean():.2f}%"}
        for flag, lab in [(True, "with near-duplicate in train/val"), (False, "without")]:
            sub = pr[pr["leaked"] == flag]
            row[lab] = f"{int(sub['ok'].sum())}/{len(sub)} = {100*sub['ok'].mean():.2f}%" if len(sub) else "-"
            leak_text.append(f"- {name} | {lab}: {int(sub['ok'].sum())}/{len(sub)} correct" +
                             (f" = {repr(float(sub['ok'].mean()))}" if len(sub) else ""))
        leak_text.append(f"- {name} | all: {int(pr['ok'].sum())}/{len(pr)} = {repr(float(pr['ok'].mean()))}")
        leak_rows.append(row)
LEAK_TABLE = pd.DataFrame(leak_rows)
print("Leakage experiment (pooled over folds):"); display(LEAK_TABLE)

naive_counts = "\n".join(f"- repeat {r} fold {f}: {int(a)} of {int(b)} test images have a near-duplicate in train/val"
                         for (r, f), (a, b) in lk.iterrows())
write_research_log(ABL_NAMES + [c.name for c in LEAKAGE] + ["convnext_base__B"], NOTEBOOK, extra_sections=[
    ("Naive split: test images with a near-duplicate in the same fold's train/val", naive_counts),
    ("Group-aware split: same count", f"{int(sum(LEAK_GROUP.values()))} test images (all folds)"),
    ("Leakage experiment: accuracy on test images with / without a near-duplicate in train/val", "\n".join(leak_text)),
])
make_results_zip(ABL_NAMES + [c.name for c in LEAKAGE], NOTEBOOK, SUMMARY)

,configuration,folds,accuracy,macro_precision,macro_recall,macro_f1,balanced_accuracy,cohen_kappa,recall Alternaria,recall Healthy,recall Leaf spot,recall Rosette,recall Rust,pooled correct,min/fold
0,resnet50__P,5,97.90 ± 0.65,97.85 ± 0.99,97.90 ± 0.87,97.83 ± 0.62,97.90 ± 0.87,97.14 ± 0.86,96.9,99.5,96.5,96.7,100.0,1684/1720,3.8
1,resnet50__P-no_aug,5,95.92 ± 1.69,95.97 ± 2.50,95.88 ± 1.85,95.77 ± 2.18,95.88 ± 1.85,94.43 ± 2.27,94.2,98.3,94.2,96.7,96.2,1650/1720,2.4
2,resnet50__P-no_imbalance,5,98.07 ± 0.77,98.53 ± 0.48,97.74 ± 0.98,98.09 ± 0.59,97.74 ± 0.98,97.36 ± 1.05,97.5,99.5,96.9,96.7,98.0,1687/1720,3.4
3,resnet50__P-loss_weights_only,5,97.27 ± 1.55,96.84 ± 2.67,97.21 ± 1.28,96.91 ± 1.99,97.21 ± 1.28,96.29 ± 2.07,95.3,97.8,98.3,96.7,98.0,1673/1720,3.4
4,resnet50__P-sampler_only,5,96.82 ± 1.33,96.33 ± 2.27,96.95 ± 1.22,96.51 ± 1.51,96.95 ± 1.22,95.67 ± 1.78,95.8,99.9,93.2,96.7,99.2,1665/1720,3.3
5,resnet50__P-no_scheduler,5,97.52 ± 1.17,97.88 ± 1.18,97.11 ± 1.76,97.39 ± 1.23,97.11 ± 1.76,96.61 ± 1.58,96.8,99.5,96.0,93.3,100.0,1677/1720,3.3
6,resnet50__P-cosine,5,98.32 ± 0.45,98.24 ± 0.88,98.21 ± 0.89,98.19 ± 0.52,98.21 ± 0.89,97.71 ± 0.59,96.9,99.5,98.0,96.7,100.0,1691/1720,3.3
7,resnet50__P-full_finetune,5,98.01 ± 0.68,98.49 ± 0.65,98.13 ± 1.09,98.28 ± 0.75,98.13 ± 1.09,97.27 ± 0.94,97.3,99.8,95.8,97.8,100.0,1686/1720,3.8
8,resnet50__P-linear_head,5,97.41 ± 1.37,97.70 ± 1.35,97.76 ± 0.97,97.69 ± 1.13,97.76 ± 0.97,96.46 ± 1.87,96.0,98.8,96.3,97.8,100.0,1676/1720,3.3
9,resnet50__vanilla_TL,5,97.16 ± 0.86,97.63 ± 1.00,96.98 ± 0.88,97.27 ± 0.92,96.98 ± 0.88,96.10 ± 1.19,95.8,99.3,95.2,95.6,99.0,1671/1720,2.2


Ablation: change relative to resnet50__P (negative = removing the component hurts)


,configuration,folds,accuracy mean diff (pts),SD (pts),folds better / worse / equal,macro_f1 mean diff (pts),SD (pts),folds better / worse / equal
0,resnet50__P-no_aug,5,-1.98,1.39,0 / 4 / 1,-2.07,2.05,1 / 4 / 0
1,resnet50__P-no_imbalance,5,+0.16,0.63,3 / 1 / 1,+0.25,0.87,4 / 1 / 0
2,resnet50__P-loss_weights_only,5,-0.63,0.99,0 / 2 / 3,-0.92,1.57,2 / 3 / 0
3,resnet50__P-sampler_only,5,-1.08,0.86,0 / 4 / 1,-1.32,0.96,0 / 5 / 0
4,resnet50__P-no_scheduler,5,-0.39,0.78,2 / 3 / 0,-0.44,0.98,1 / 4 / 0
5,resnet50__P-cosine,5,+0.42,0.45,3 / 0 / 2,+0.35,0.49,3 / 2 / 0
6,resnet50__P-full_finetune,5,+0.11,1.12,2 / 3 / 0,+0.45,1.30,2 / 3 / 0
7,resnet50__P-linear_head,5,-0.49,1.46,2 / 3 / 0,-0.14,1.56,2 / 3 / 0
8,resnet50__vanilla_TL,5,-0.75,1.10,1 / 3 / 1,-0.57,1.26,2 / 3 / 0


Accuracy (%) under simulated degradations:


configuration,resnet50__P,resnet50__P-no_aug,resnet50__P-no_imbalance,resnet50__P-loss_weights_only,resnet50__P-sampler_only,resnet50__P-no_scheduler,resnet50__P-cosine,resnet50__P-full_finetune,resnet50__P-linear_head,resnet50__vanilla_TL
clean,97.9,95.9,98.1,97.3,96.8,97.5,98.3,98.0,97.4,97.2
dark_0.6,97.0,92.0,97.3,96.3,96.7,96.9,97.4,97.8,96.6,90.3
overexposed_1.4,96.6,85.8,96.1,95.7,95.0,96.3,96.7,97.4,95.7,91.0
low_contrast_0.6,96.6,93.3,97.1,96.7,96.7,97.0,97.7,97.9,97.1,92.0
hard_shadow_0.4,94.8,89.6,95.6,95.9,95.0,94.0,96.0,97.0,95.0,89.0
blur_sigma2,96.3,90.8,97.2,96.1,95.7,95.9,96.6,96.7,95.1,91.8
noise_sd0.05,97.3,94.2,98.1,95.1,96.0,97.4,96.9,97.5,95.6,95.9
jpeg_q20,98.1,94.4,97.9,97.0,97.2,97.6,98.4,98.4,97.0,96.5
occlusion_25pct,87.9,74.2,91.1,88.7,88.2,85.4,89.7,90.6,88.2,79.9
rotation_45,97.6,93.5,97.9,97.5,96.9,97.4,98.1,97.9,97.2,95.8


Leakage experiment (pooled over folds):


,configuration,split,all test images,with near-duplicate in train/val,without
0,resnet50__P,group-aware,1684/1720 = 97.91%,-,1684/1720 = 97.91%
1,resnet50__P__naive_split,naive,1705/1720 = 99.13%,905/910 = 99.45%,800/810 = 98.77%
2,convnext_base__B,group-aware,1708/1720 = 99.30%,-,1708/1720 = 99.30%
3,convnext_base__B__naive_split,naive,1709/1720 = 99.36%,910/910 = 100.00%,799/810 = 98.64%


research log written: /content/drive/MyDrive/Peanut_Revision/09_logs/research_log_NB03.md
Saved /content/drive/MyDrive/Peanut_Revision/04_results/NB03_results.zip (0.9 MB)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>